# Recuperacion completa de la subclave final de SPN16

Este notebook recupera los cuatro nibbles de la subclave final $K_5$ de un SPN16 de cuatro rondas.

La caracteristica seleccionada fue calculada previamente con `find_characteristics/spn16_characteristic_search.py`:

$$\mathtt{0F00}\longrightarrow\mathtt{0400}\longrightarrow\mathtt{0440}\longrightarrow\mathtt{4264}.$$

Su probabilidad es

$$P=\frac{27}{2048}=0.01318359375,$$

y su peso diferencial es

$$w=-\log_2(P)\approx 6.2451.$$

Se eligio esta trayectoria porque `4264` activa los cuatro nibbles antes de la ultima capa de S-boxes. Por tanto, un solo experimento permite obtener candidatos para los cuatro nibbles de $K_5$.

## 1. Que significa recuperacion completa

En este notebook, recuperacion completa significa obtener los 16 bits de la subclave final

$$K_5=(k_0,k_1,k_2,k_3).$$

La implementacion de SPN16 divide una clave maestra de 80 bits en cinco subclaves independientes de 16 bits. Por ello, conocer $K_5$ no determina automaticamente $K_1,K_2,K_3,K_4$.

La recuperacion de la subclave final completa es la etapa estandar del ataque diferencial y reduce el espacio desconocido de clave en 16 bits.

## 2. Idea matematica

En el nibble $j$ de la ultima ronda se tiene

$$C_j=S(U_j)\oplus K_{5,j}.$$

Para un candidato $k$ se calcula

$$\widehat U_j(k)=S^{-1}(C_j\oplus k).$$

Dado un par $(C,C')$, el candidato recibe un punto si

$$\widehat U_j(k)\oplus\widehat U'_j(k)=\Delta U_j.$$

El score es

$$\operatorname{score}_j(k)=\sum_{i=1}^{N}\mathbf 1\left[\widehat U_{i,j}(k)\oplus\widehat U'_{i,j}(k)=\Delta U_j\right].$$

Como

$$\Delta U=\mathtt{4264},$$

se prueban las condiciones

$$\Delta U_0=4,\quad\Delta U_1=2,\quad\Delta U_2=6,\quad\Delta U_3=4.$$

In [ ]:
from pathlib import Path
import json
import math
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd


def add_repo_root_to_path():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "cryptosystems").is_dir() and (path / "find_characteristics").is_dir():
            if str(path) not in sys.path:
                sys.path.insert(0, str(path))
            return path
    raise RuntimeError("No se encontro la raiz del repositorio")


ROOT = add_repo_root_to_path()
from cryptosystems import spn16

pd.set_option("display.max_columns", None)
print(f"Repo root: {ROOT}")

## 3. Lectura de la caracteristica guardada

La trayectoria se lee desde un archivo JSON de `find_characteristics`. Esto permite conservar los parametros con los que fue encontrada.

In [ ]:
characteristic_path = ROOT / "find_characteristics" / "spn16_full_key_recovery_characteristic.json"
characteristic_data = json.loads(characteristic_path.read_text(encoding="utf-8"))
selected_characteristic = characteristic_data["characteristic"]

TRAIL = [int(value, 16) for value in selected_characteristic["deltas"]]
TRAIL_PROBABILITY = float(selected_characteristic["probability"])
TRAIL_WEIGHT = float(selected_characteristic["weight"])

print("Trayectoria:")
print(" -> ".join(selected_characteristic["deltas"]))
print(f"Probabilidad: {TRAIL_PROBABILITY:.12f}")
print(f"Peso: {TRAIL_WEIGHT:.4f}")

## 4. Parametros del experimento

Se utilizan 20000 pares elegidos para reducir las fluctuaciones estadísticas. La clave real se usa unicamente para construir el oraculo y verificar el resultado al final.

In [ ]:
ROUNDS = 4
MASTER_KEY_HEX = "00112233445566778899"

DELTA_P = TRAIL[0]
DELTA_U = TRAIL[-1]
N_PAIRS = 20000
SEED = 2026


def extract_nibble(block: int, position: int) -> int:
    shift = 4 * (3 - position)
    return (block >> shift) & 0xF


EXPECTED_DELTA_BY_POSITION = {
    position: extract_nibble(DELTA_U, position)
    for position in range(4)
}

subkeys = spn16.expand_key_from_hex(MASTER_KEY_HEX, ROUNDS)


def oracle_encrypt(plaintext: int) -> int:
    return spn16.spn_encrypt_block(plaintext, subkeys, ROUNDS)


print(f"Rondas: {ROUNDS}")
print(f"Delta P: 0x{DELTA_P:04X}")
print(f"Delta U: 0x{DELTA_U:04X}")
print(f"Deltas por posicion: {EXPECTED_DELTA_BY_POSITION}")
print(f"Numero de pares: {N_PAIRS}")

## 5. Generacion de pares elegidos

Para cada texto claro $P$ se construye

$$P'=P\oplus\mathtt{0F00}.$$

Los dos textos se cifran con el mismo oraculo.

In [ ]:
rng = random.Random(SEED)
plaintext_pairs = []
ciphertext_pairs = []

for _ in range(N_PAIRS):
    p = rng.randrange(0, 1 << 16)
    p_prime = p ^ DELTA_P
    c = oracle_encrypt(p)
    c_prime = oracle_encrypt(p_prime)
    plaintext_pairs.append((p, p_prime))
    ciphertext_pairs.append((c, c_prime))

pd.DataFrame([
    {
        "P": f"0x{p:04X}",
        "P'": f"0x{p_prime:04X}",
        "P XOR P'": f"0x{p ^ p_prime:04X}",
        "C": f"0x{c:04X}",
        "C'": f"0x{c_prime:04X}",
    }
    for (p, p_prime), (c, c_prime) in zip(plaintext_pairs[:8], ciphertext_pairs[:8])
])

## 6. Prueba de candidatos sobre un par

La tabla muestra los 16 candidatos para el primer nibble y el primer par. Un solo par puede aceptar varios candidatos; la separación aparece al acumular toda la muestra.

In [ ]:
EXAMPLE_POSITION = 0
first_c, first_c_prime = ciphertext_pairs[0]
first_c_nibble = extract_nibble(first_c, EXAMPLE_POSITION)
first_c_prime_nibble = extract_nibble(first_c_prime, EXAMPLE_POSITION)
example_delta = EXPECTED_DELTA_BY_POSITION[EXAMPLE_POSITION]

single_pair_rows = []
for key_guess in range(16):
    u_hat = spn16.S_BOX_INV[first_c_nibble ^ key_guess]
    u_hat_prime = spn16.S_BOX_INV[first_c_prime_nibble ^ key_guess]
    delta_hat = u_hat ^ u_hat_prime
    single_pair_rows.append({
        "candidato k": f"0x{key_guess:X}",
        "U estimado": f"0x{u_hat:X}",
        "U' estimado": f"0x{u_hat_prime:X}",
        "Delta estimado": f"0x{delta_hat:X}",
        "Delta objetivo": f"0x{example_delta:X}",
        "cumple": delta_hat == example_delta,
    })

pd.DataFrame(single_pair_rows)

## 7. Conteo para los cuatro nibbles

Se prueban 16 candidatos en cada posicion. El mejor candidato de cada ranking se incorpora a la subclave recuperada.

In [ ]:
def score_candidate(key_guess: int, position: int, pairs) -> int:
    target_delta = EXPECTED_DELTA_BY_POSITION[position]
    score = 0

    for c, c_prime in pairs:
        c_nibble = extract_nibble(c, position)
        c_prime_nibble = extract_nibble(c_prime, position)
        u_hat = spn16.S_BOX_INV[c_nibble ^ key_guess]
        u_hat_prime = spn16.S_BOX_INV[c_prime_nibble ^ key_guess]
        if (u_hat ^ u_hat_prime) == target_delta:
            score += 1
    return score


scores_by_position = {}
for position in range(4):
    rows = []
    for key_guess in range(16):
        rows.append({
            "candidate_int": key_guess,
            "candidato": f"0x{key_guess:X}",
            "score": score_candidate(key_guess, position, ciphertext_pairs),
        })
    scores_by_position[position] = pd.DataFrame(rows).sort_values(
        ["score", "candidate_int"], ascending=[False, True]
    ).reset_index(drop=True)

top_candidates_df = pd.concat([
    df.head(5).assign(posicion=position, ranking=range(1, 6))
    for position, df in scores_by_position.items()
])[["posicion", "ranking", "candidato", "score"]]

top_candidates_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)

for ax, position in zip(axes.flat, range(4)):
    plot_df = scores_by_position[position].sort_values("candidate_int")
    best_guess = int(scores_by_position[position].iloc[0]["candidate_int"])
    colors = [
        "#2A9D8F" if value == best_guess else "#8D99AE"
        for value in plot_df["candidate_int"]
    ]
    ax.bar(plot_df["candidato"], plot_df["score"], color=colors)
    ax.set_title(f"Posicion {position}: Delta U={EXPECTED_DELTA_BY_POSITION[position]:X}")
    ax.set_xlabel("Candidato k")
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.2)

fig.suptitle("Conteo de candidatos para K5")
plt.tight_layout()
plt.show()

## 8. Construccion de la subclave completa

Los cuatro candidatos ganadores se concatenan en orden de nibble.

In [ ]:
recovered_nibbles = [
    int(scores_by_position[position].iloc[0]["candidate_int"])
    for position in range(4)
]
recovered_final_key = (
    (recovered_nibbles[0] << 12)
    | (recovered_nibbles[1] << 8)
    | (recovered_nibbles[2] << 4)
    | recovered_nibbles[3]
)
true_final_key = subkeys[-1]

result_rows = []
for position in range(4):
    df = scores_by_position[position]
    best = df.iloc[0]
    second = df.iloc[1]
    true_nibble = extract_nibble(true_final_key, position)
    result_rows.append({
        "posicion": position,
        "Delta U": f"0x{EXPECTED_DELTA_BY_POSITION[position]:X}",
        "recuperado": f"0x{int(best['candidate_int']):X}",
        "real": f"0x{true_nibble:X}",
        "mejor score": int(best["score"]),
        "segundo score": int(second["score"]),
        "margen": int(best["score"] - second["score"]),
        "correcto": int(best["candidate_int"]) == true_nibble,
    })

print(f"K5 real:        0x{true_final_key:04X}")
print(f"K5 recuperada:  0x{recovered_final_key:04X}")
print(f"Recuperacion completa correcta: {recovered_final_key == true_final_key}")
pd.DataFrame(result_rows)

## 9. Verificacion de la diferencia completa

Con la subclave recuperada se elimina el whitening final y se aplica $S^{-1}$ a los cuatro nibbles. Se cuenta cuantos pares reconstruyen exactamente

$$\Delta U=\mathtt{4264}.$$

La cantidad esperada es aproximadamente $N\cdot P$.

In [ ]:
right_pair_count = 0
for c, c_prime in ciphertext_pairs:
    u = spn16._inv_sub_bytes16(c ^ recovered_final_key)
    u_prime = spn16._inv_sub_bytes16(c_prime ^ recovered_final_key)
    if (u ^ u_prime) == DELTA_U:
        right_pair_count += 1

expected_right_pairs = N_PAIRS * TRAIL_PROBABILITY

print(f"Pares que reconstruyen 0x{DELTA_U:04X}: {right_pair_count}")
print(f"Proporcion observada: {right_pair_count / N_PAIRS:.6f}")
print(f"Cantidad esperada N*P: {expected_right_pairs:.2f}")
print(f"Probabilidad teorica: {TRAIL_PROBABILITY:.6f}")

## 10. Efecto del numero de pares

Se estudia el rango del nibble real y su margen respecto al mejor candidato incorrecto.

In [ ]:
sample_sizes = [500, 1000, 2000, 4000, 10000, 20000]
stability_rows = []

for position in range(4):
    true_nibble = extract_nibble(true_final_key, position)
    for sample_size in sample_sizes:
        sample_pairs = ciphertext_pairs[:sample_size]
        sample_scores = {
            key_guess: score_candidate(key_guess, position, sample_pairs)
            for key_guess in range(16)
        }
        ranking = sorted(sample_scores, key=lambda key_guess: sample_scores[key_guess], reverse=True)
        best_wrong = max(score for key_guess, score in sample_scores.items() if key_guess != true_nibble)
        stability_rows.append({
            "posicion": position,
            "pares": sample_size,
            "rango clave real": ranking.index(true_nibble) + 1,
            "score clave real": sample_scores[true_nibble],
            "mejor score incorrecto": best_wrong,
            "margen": sample_scores[true_nibble] - best_wrong,
        })

stability_df = pd.DataFrame(stability_rows)
stability_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for ax, position in zip(axes.flat, range(4)):
    position_df = stability_df[stability_df["posicion"] == position]
    ax.plot(position_df["pares"], position_df["margen"], marker="o", color="#2A9D8F")
    ax.axhline(0, color="#D62828", linestyle="--")
    ax.set_title(f"Posicion {position}")
    ax.set_xlabel("Numero de pares")
    ax.set_ylabel("Margen de la clave real")
    ax.grid(alpha=0.25)

fig.suptitle("Estabilidad de los cuatro nibbles recuperados")
plt.tight_layout()
plt.show()

## 11. Lectura de los resultados

Los resultados muestran que:

1. la diferencia `4264` proporciona informacion sobre los cuatro nibbles de $K_5$;
2. cada posicion se resuelve con un ranking independiente de 16 candidatos;
3. la concatenacion de los cuatro ganadores produce `K5 = 8899`;
4. al invertir la ultima ronda con la clave recuperada, la frecuencia observada de `4264` es cercana a la probabilidad teorica de la caracteristica;
5. el primer nibble necesita mas pares para separarse claramente de candidatos incorrectos.

El costo principal es

$$4\cdot 2^4\cdot N=64N$$

pruebas candidato-par.

## 12. Texto base para una tesis

> Se selecciono la caracteristica diferencial `0F00 -> 0400 -> 0440 -> 4264`, con probabilidad $27/2048$ y peso aproximado 6.2451, debido a que su diferencia final activa los cuatro nibbles en la entrada de la ultima capa de sustitucion. Se generaron pares elegidos de textos claros con diferencia `0F00` y se cifraron mediante un oraculo SPN16 de cuatro rondas. Para cada posicion de la subclave final se probaron los dieciseis candidatos posibles. Bajo cada hipotesis se elimino el XOR de clave y se aplico la S-box inversa. El candidato recibio un punto cuando la diferencia reconstruida coincidio con el nibble correspondiente de `4264`. Los cuatro candidatos con mayor conteo se concatenaron, obteniendo la subclave final completa `8899`.

Al documentar el experimento conviene reportar:

- la trayectoria, probabilidad y peso;
- el criterio de seleccion de una diferencia con cuatro nibbles activos;
- el numero de pares y la semilla;
- los rankings y margenes por posicion;
- la subclave final recuperada;
- la cantidad observada de pares que reconstruyen la diferencia completa.

## 13. Limitaciones

- El ataque recupera completamente $K_5$, no los 80 bits de la clave maestra.
- El calendario de claves de este SPN16 toma bloques independientes de la clave maestra, por lo que $K_1$ a $K_4$ no se derivan de $K_5$.
- El éxito depende de la calidad de la caracteristica y de usar suficientes pares.
- La característica fue obtenida mediante beam search con poda; es la mejor encontrada bajo los parametros guardados, no una prueba de optimalidad global.